In [1]:
from dotenv import load_dotenv
from langchain.tools import ToolRuntime, tool
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from uuid import uuid4
from langgraph.types import Command
from pprint import pprint

In [2]:
load_dotenv()

True

In [3]:
class EmailState(AgentState):
    email: str

In [4]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    print(body)
    return "Email sent"

In [5]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [6]:
config = {"configurable": {"thread_id": str(uuid4())}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [7]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          "That's "
                                                                          'no '
                                                                          'problem, '
                                                                          "let's "
                                                                          'reschedule. '
  

In [8]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, thanks for letting me know. That's no problem, let's reschedule. What time works for you? Best, Seán.


In [9]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": (
                        "Don't Send the email. Rewrite the email in a dissapointed tone "
                        "with a warning. Send the email with the rewritten draft."
                    )
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
)  

In [10]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'I '
                                                                          'am '
                                                                          'disappointed '
                                                                          'to '
                                                                          'hear '
                                                                          'that '
                                                                          "you'll "
                                                                          'be '
                                                                          'late '
                                                                          'for '
                 

In [11]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, I am disappointed to hear that you'll be late for our meeting. It is quite short notice to reschedule. Please ensure this doesn't become a habit, as I have a busy schedule. Let me know when you are actually available. Regards, Seán.


In [12]:
response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config
)

Hi John, I am disappointed to hear that you'll be late for our meeting. It is quite short notice to reschedule. Please ensure this doesn't become a habit, as I have a busy schedule. Let me know when you are actually available. Regards, Seán.


In [13]:
pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='71f7facc-3822-4b03-be9e-d54071fd1a62'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "me"}'}, '__gemini_function_call_thought_signatures__': {'call_869147': 'EnMKcQFpFH0To+8neUiH+qxqQD96dBdMvnAdhreyR4hPR2mPk/5SSS2Ea8M0ycxuwd0eqB3dAVf6YRtFroYbhEXgJao2xLxhaHIquyc29Kd76aPvkui0+VlW+aeyviWMszzFxCauUxlJQRX0Vkp10di0wSLQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1357b-e46e-7563-88bc-5d6b919f64f9-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'me'}, 'id': 'call_869147', 'type': 'tool_call'}], invalid_tool_c